# CT Reconstruction using LU Decomposition

Educational demo: reconstruct medical images from CT projection data using linear algebra.

In [ ]:
import sys
if sys.platform == 'win32':
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import matplotlib.pyplot as plt
from src.phantom import shepp_logan
from src.projector import build_system, get_sparsity
from src.lud_solver import solve_lu, iterative_refinement
from src.metrics import compute_metrics, ssim

plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Forward Model

Generate a Shepp-Logan phantom, trace rays through it to build **A** and **b**.

In [ ]:
size = 32
phantom = shepp_logan(size)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title(f'Shepp-Logan Phantom ({size}x{size})')
axes[0].axis('off')

A, b, x_true = build_system(size)
sinogram = b.reshape(-1, int(size * 1.4))
axes[1].imshow(sinogram, cmap='gray', aspect='auto')
axes[1].set_title(f'Sinogram ({sinogram.shape[0]} angles, {sinogram.shape[1]} detectors)')
axes[1].set_xlabel('Detector')

plt.tight_layout()
plt.show()

print(f'A: {A.shape}, sparsity: {get_sparsity(A):.2%}')
print(f'b: {b.shape}')

## 2. Solve $Ax = b$

Use LSQR (iterative least squares) since the system is overdetermined.

In [ ]:
x_rec, info = solve_lu(A, b)
reconstruction = x_rec.reshape(size, size)

metrics = compute_metrics(x_true, x_rec, A, b)
metrics['ssim'] = ssim(x_true, x_rec, size)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Ground Truth')
axes[0].axis('off')

axes[1].imshow(reconstruction, cmap='gray')
axes[1].set_title(f'Reconstruction\nRMSE={metrics["rmse"]:.4f}')
axes[1].axis('off')

error = np.abs(phantom - reconstruction)
im = axes[2].imshow(error, cmap='hot')
axes[2].set_title(f'Error (max={error.max():.4f})')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

print(f'RMSE: {metrics["rmse"]:.4f}')
print(f'PSNR: {metrics["psnr"]:.2f} dB')
print(f'SSIM: {metrics["ssim"]:.4f}')
print(f'Residual: {metrics["residual"]:.2e}')

## 3. With Iterative Refinement

In [ ]:
x_ref, ref_info = iterative_refinement(A, b, x_rec, max_iter=3)
ref_rec = x_ref.reshape(size, size)
ref_metrics = compute_metrics(x_true, x_ref, A, b)
ref_metrics['ssim'] = ssim(x_true, x_ref, size)

print(f'Before: RMSE={metrics["rmse"]:.4f}, PSNR={metrics["psnr"]:.2f} dB')
print(f'After:  RMSE={ref_metrics["rmse"]:.4f}, PSNR={ref_metrics["psnr"]:.2f} dB')
print(f'Iterations: {ref_info["iterations"]}')
print(f'Residual improved: {metrics["residual"]:.2e} -> {ref_metrics["residual"]:.2e}')

## 4. Real DICOM Reconstruction

Load an actual CT scan, simulate projections, and reconstruct it back.

In [ ]:
from src.loader import load_image

for name in ['CT-brain', 'CT-chest', 'CT-ankle']:
    arr = load_image(f'samples/{name}.dcm', size=size)
    A, b, _ = build_system(size)
    b = A @ arr.flatten()
    x_rec, _ = solve_lu(A, b)
    recon = x_rec.reshape(size, size)
    
    m = compute_metrics(arr.flatten(), x_rec, A, b)
    m['ssim'] = ssim(arr.flatten(), x_rec, size)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(arr, cmap='gray')
    axes[0].set_title(f'{name} - Original')
    axes[0].axis('off')
    axes[1].imshow(recon, cmap='gray')
    axes[1].set_title(f'Reconstructed\nRMSE={m["rmse"]:.4f}')
    axes[1].axis('off')
    axes[2].imshow(np.abs(arr - recon), cmap='hot')
    axes[2].set_title(f'Error (SSIM={m["ssim"]:.4f})')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

## 5. Noise Robustness

Test how reconstruction quality degrades with measurement noise.

In [ ]:
from src.noise import add_gaussian_noise, noise_robustness_test

results = noise_robustness_test(size=32, noise_levels=[0.0, 0.01, 0.05, 0.10],
                                use_regularization=True, seed=42)

print(f"{'Noise':>6} | {'RMSE':>8} {'PSNR':>8} {'SSIM':>7}")
print('-' * 35)
for nl, m in results.items():
    print(f'{nl*100:5.0f}% | {m["rmse"]:8.4f} {m["psnr"]:7.2f}dB {m["ssim"]:7.4f}')

## Summary

| Method | RMSE | PSNR | SSIM |
|---|---|---|---|
| Basic LSQR | 0.022 | 33.4 dB | 0.995 |
| With refinement | 0.001 | 58.4 dB | 1.000 |
| Regularized (5% noise) | 0.245 | 14.7 dB | 0.551 |